In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
bronze_path = "abfss://bronze@ecommercestoragev2.dfs.core.windows.net/customers"
silver_path = "abfss://silver@ecommercestoragev2.dfs.core.windows.net/customers"

hlp_path = f"{silver_path}/customer_hlp"
dim_path = f"{silver_path}/customer_dim"

batch_id = "0"

In [0]:
silver_df = (
    spark.read.format("parquet").load(bronze_path)
    .select("id", "first_name", "last_name", "email", "cnic")
    .distinct()
    .withColumnRenamed("id", "customer_id")
)

hlp_df = (
    spark.read.format("delta").load(hlp_path)
    .select("customer_id", "customer_sk")
)

In [0]:
source_df = (
    silver_df
    .join(hlp_df, on="customer_id", how="left")
    .filter(F.col("customer_sk").isNotNull())
    .select(
        F.col("customer_sk"),
        F.col("first_name"),
        F.col("last_name"),
        F.col("email"),
        F.col("cnic"),
        F.current_timestamp().alias("created_at"),   # derived
        F.current_timestamp().alias("updated_at"),   # derived
        F.lit("customers").alias("source"),
        F.lit("I").alias("oper"),
        F.current_timestamp().alias("ins_tmstmp"),
        F.current_timestamp().alias("upd_tmstmp"),
        F.lit(batch_id).alias("batch_id"),
    )
)

In [0]:
if not DeltaTable.isDeltaTable(spark, dim_path):
    # First run — write directly
    (
        source_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(dim_path)
    )
    print("Bootstrap: customer_dim created.")

else:
    dim_table = DeltaTable.forPath(spark, dim_path)

    (
        dim_table.alias("target")
        .merge(
            source_df.alias("source"),
            "target.customer_sk = source.customer_sk"
        )
        .whenMatchedUpdate(set={
            "first_name":  "source.first_name",
            "last_name":   "source.last_name",
            "email":       "source.email",
            "cnic":        "source.cnic",
            "updated_at":  "source.updated_at",
            "oper":        F.lit("U"),
            "upd_tmstmp":  "source.upd_tmstmp",
            "batch_id":    "source.batch_id",
        })
        .whenNotMatchedInsert(values={
            "customer_sk":  "source.customer_sk",
            "first_name":   "source.first_name",
            "last_name":    "source.last_name",
            "email":        "source.email",
            "cnic":         "source.cnic",
            "created_at":   "source.created_at",
            "updated_at":   "source.updated_at",
            "source":       "source.source",
            "oper":         "source.oper",
            "ins_tmstmp":   "source.ins_tmstmp",
            "upd_tmstmp":   "source.upd_tmstmp",
            "batch_id":     "source.batch_id",
        })
        .execute()
    )
    print("Incremental: customer_dim merged.")